# QUERY 3
Cuenta de origen y monto de transacciones USD en el período [2022-09-06, 2022-09-15]
con monto menor a 1 centésimo del promedio encontrado para el mismo formato de
pago en el período [2022-09-01, 2022-09-05]

In [ ]:
import pandas as pd

RUTA_DE_DATASETS             = "../../datasets"
NOMBRE_ARCHIVO_COMPLETO      = "LI-Small_Trans.csv"
NOMBRE_ARCHIVO_TRANSACCIONES = "q3_sample.csv"
RUTA_RESULTADO_QUERY3        = "q3_solucion.csv"
SAMPLE_N                     = 50000  

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_COMPLETO}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

transacciones = transacciones_completas.sample(n=min(SAMPLE_N, len(transacciones_completas)), random_state=42)
guardar_csv(transacciones, f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}")
print(f"Sample generada: {len(transacciones)} filas → {NOMBRE_ARCHIVO_TRANSACCIONES}")

Sample generada: 50000 filas → q3_sample.csv


In [18]:
import pandas as pd

guardar_csv = lambda df, ruta: df.to_csv(ruta, index=False)

transacciones_completas = pd.read_csv(
    f"{RUTA_DE_DATASETS}/{NOMBRE_ARCHIVO_TRANSACCIONES}",
    dtype={"From Bank": "string", "Account": "string",
           "To Bank": "string", "Account.1": "string"}
)

# Sample para desarrollo
SAMPLE_N = 5000
transacciones = transacciones_completas.sample(n=min(SAMPLE_N, len(transacciones_completas)), random_state=42)
print(f"Modo SAMPLE: {len(transacciones)} transacciones")

Modo SAMPLE: 5000 transacciones


In [19]:
# Filtro USD + parseo de tipos
transacciones_usd = transacciones[
    transacciones["Payment Currency"] == "US Dollar"
].copy()

transacciones_usd["Timestamp"]    = pd.to_datetime(transacciones_usd["Timestamp"])
transacciones_usd["Amount Paid"]  = pd.to_numeric(transacciones_usd["Amount Paid"], errors="coerce")
transacciones_usd = transacciones_usd.dropna(subset=["Amount Paid", "Payment Format"])

# Separación por período
periodo_temprano = transacciones_usd[
    (transacciones_usd["Timestamp"] >= "2022-09-01") &
    (transacciones_usd["Timestamp"] <= "2022-09-05")
]
periodo_tardio = transacciones_usd[
    (transacciones_usd["Timestamp"] >= "2022-09-06") &
    (transacciones_usd["Timestamp"] <= "2022-09-15")
]

print(f"Período temprano (1/9-5/9): {len(periodo_temprano)} transacciones USD")
print(f"Período tardío  (6/9-15/9): {len(periodo_tardio)} transacciones USD")

Período temprano (1/9-5/9): 853 transacciones USD
Período tardío  (6/9-15/9): 829 transacciones USD


In [20]:
# Promedio por formato — simula el dict {formato: {suma, count}} -> promedio
stats_por_formato = (
    periodo_temprano
    .groupby("Payment Format")["Amount Paid"]
    .agg(suma="sum", count="count")
)
stats_por_formato["promedio"] = stats_por_formato["suma"] / stats_por_formato["count"]

print(stats_por_formato)

                        suma  count       promedio
Payment Format                                    
ACH             2.804402e+06     90   31160.021889
Cash            7.985338e+06     65  122851.348923
Cheque          2.027226e+08    267  759260.777940
Credit Card     5.950006e+05    203    2931.037586
Reinvestment    2.051421e+07    204  100559.875245
Wire            2.361446e+06     24   98393.597083


In [21]:
# Filtro: monto < 1% del promedio del mismo formato en período temprano
df = periodo_tardio.copy()
df["Promedio Formato"] = df["Payment Format"].map(stats_por_formato["promedio"])

# Formatos sin promedio en período temprano → se descartan
df = df.dropna(subset=["Promedio Formato"])

resultado_query3 = df[
    df["Amount Paid"] < df["Promedio Formato"] * 0.01
][["Account", "Amount Paid"]].rename(columns={"Account": "From Account"}).reset_index(drop=True)

guardar_csv(resultado_query3, RUTA_RESULTADO_QUERY3)
print(f"Resultados Q3: {len(resultado_query3)} transacciones")
resultado_query3.head()

Resultados Q3: 394 transacciones


,From Account,Amount Paid
0,811537E60,266.37
1,10042B660,445.56
2,8019B6DD0,72.38
3,800D9D420,182.10
4,812D0EA70,749.14
